# Loss Functions

## Learning Objectives
1. Understand the statistical assumptions behind different loss functions and their robustness properties.
2. Implement MSE, MAE, Huber, BCE, and Cross-Entropy from scratch in NumPy.
3. Compare regression vs classification losses and outlier sensitivity using PyTorch with OOM handling.
4. Apply focal loss for class imbalance, contrastive loss for metric learning, and compound loss with learnable weights in production settings.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Loss Functions from Scratch (NumPy)

In [ ]:
# Implementing core loss functions in NumPy clarifies the mathematical
# relationship between each loss and the underlying statistical assumption:
# MSE assumes Gaussian errors, MAE assumes Laplace errors (robust to outliers),
# Huber blends both, BCE assumes Bernoulli, CE assumes categorical distribution.

def mse_loss(y_true, y_pred):
    '''Mean Squared Error — penalises large errors quadratically.
    MLE estimator under Gaussian noise assumption.
    '''
    return np.mean((y_true - y_pred) ** 2)


def mae_loss(y_true, y_pred):
    '''Mean Absolute Error — linear penalty; robust to outliers.
    MLE estimator under Laplace distribution assumption.
    '''
    return np.mean(np.abs(y_true - y_pred))


def huber_loss(y_true, y_pred, delta=1.0):
    '''Huber Loss — quadratic for |e| < delta, linear beyond.
    Best of both worlds: smooth near 0, robust to outliers.
    '''
    residual = np.abs(y_true - y_pred)
    return np.where(residual <= delta,
                   0.5 * residual ** 2,
                   delta * residual - 0.5 * delta ** 2).mean()


def bce_loss(y_true, y_pred, eps=1e-7):
    '''Binary Cross-Entropy — for binary classification.
    Assumes Bernoulli distribution; y_pred in (0, 1).
    '''
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def cross_entropy_loss(y_true_oh, y_pred_logits):
    '''Multi-class Cross-Entropy from logits.
    y_true_oh: one-hot encoded labels, shape (n, n_classes)
    y_pred_logits: raw logits, shape (n, n_classes)
    '''
    exp_logits = np.exp(y_pred_logits - y_pred_logits.max(axis=1, keepdims=True))
    probs      = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return -np.mean(np.sum(y_true_oh * np.log(probs + 1e-9), axis=1))


# Demonstrate each loss
np.random.seed(0)
y_reg   = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred  = np.array([1.2, 1.8, 3.1, 3.9, 5.5])
y_pred_outlier = np.array([1.2, 1.8, 3.1, 3.9, 50.0])  # one large outlier

print('Regression losses (normal data):')
print(f'  MSE:   {mse_loss(y_reg, y_pred):.4f}   Outlier: {mse_loss(y_reg, y_pred_outlier):.2f}')
print(f'  MAE:   {mae_loss(y_reg, y_pred):.4f}   Outlier: {mae_loss(y_reg, y_pred_outlier):.2f}')
print(f'  Huber: {huber_loss(y_reg, y_pred):.4f}   Outlier: {huber_loss(y_reg, y_pred_outlier):.2f}')
print('Huber and MAE stay bounded; MSE explodes with outliers.')

# Classification losses
y_bin   = np.array([1, 0, 1, 1, 0], dtype=float)
p_pred  = np.array([0.9, 0.1, 0.8, 0.7, 0.2])
print(f'\nBCE loss: {bce_loss(y_bin, p_pred):.4f}')

n_cls = 4
logits_mc = np.random.randn(5, n_cls)
labels_oh = np.zeros((5, n_cls))
labels_oh[np.arange(5), np.random.randint(0, n_cls, 5)] = 1
print(f'Cross-Entropy loss: {cross_entropy_loss(labels_oh, logits_mc):.4f}')


## Level 2: Regression vs Classification Loss Comparison + Outlier Behaviour (PyTorch)

In [ ]:
# Train the same regression network with MSE, MAE, and Huber on data
# that contains outliers. Measure how each loss responds to corrupted targets.

torch.manual_seed(0)

# Regression data with 10% outliers
N = 1000
X_reg = torch.randn(N, 8, device=device)
noise  = torch.randn(N, device=device) * 0.5
y_clean = 2 * X_reg[:, 0] - X_reg[:, 1] + noise   # true signal
outlier_mask = (torch.rand(N) < 0.10)               # 10% outliers
y_noisy  = y_clean.clone()
y_noisy[outlier_mask] += torch.randn(outlier_mask.sum(), device=device) * 20

reg_ds  = TensorDataset(X_reg[:800], y_noisy[:800].unsqueeze(1))
reg_val = TensorDataset(X_reg[800:], y_clean[800:].unsqueeze(1))  # clean val
reg_trl = DataLoader(reg_ds,  batch_size=64, shuffle=True)
reg_vll = DataLoader(reg_val, batch_size=128)


def make_regressor():
    return nn.Sequential(
        nn.Linear(8, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1)
    ).to(device)


loss_fns = {
    'MSE':          nn.MSELoss(),
    'MAE':          nn.L1Loss(),
    'Huber(d=1)':   nn.HuberLoss(delta=1.0),
    'Huber(d=0.1)': nn.HuberLoss(delta=0.1),
}

loss_results = {}
for name, loss_fn in loss_fns.items():
    model_r = make_regressor()
    opt_r   = torch.optim.Adam(model_r.parameters(), lr=1e-3)
    for _ in range(40):
        model_r.train()
        for xb, yb in reg_trl:
            try:
                opt_r.zero_grad()
                loss_fn(model_r(xb), yb).backward()
                opt_r.step()
            except RuntimeError as exc:
                if 'out of memory' in str(exc).lower():
                    torch.cuda.empty_cache(); continue
                raise
    model_r.eval()
    with torch.no_grad():
        val_mse = sum(
            nn.MSELoss()(model_r(xb), yb).item() * len(xb)
            for xb, yb in reg_vll
        ) / len(reg_val)
    loss_results[name] = val_mse

print('Val MSE on clean data (lower = better, trained on noisy data):')
for n, v in loss_results.items():
    print(f'  {n:<18}: {v:.4f}')
print('Huber and MAE outperform MSE when training data contains outliers.')


## Real-World Example 1: Focal Loss for Class Imbalance

In [ ]:
# Focal Loss (Lin et al. 2017): down-weights easy examples, focuses on hard ones.
# FL(p_t) = -(1-p_t)^gamma * log(p_t)
# gamma=0 reduces to standard CE; gamma=2 is typical for imbalanced datasets.
# Used in: RetinaNet object detection, medical image classification.

torch.manual_seed(4)


class FocalLoss(nn.Module):
    '''Multi-class Focal Loss with class balancing alpha weights.
    Args:
        gamma: focusing parameter; higher = more focus on hard examples
        alpha: per-class weight tensor (handles class imbalance)
    '''
    def __init__(self, gamma: float = 2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # tensor of shape (n_classes,)

    def forward(self, logits, targets):
        ce_loss   = F.cross_entropy(logits, targets, reduction='none')
        p_t       = torch.exp(-ce_loss)   # confidence in correct class
        focal_w   = (1 - p_t) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_w = focal_w * alpha_t
        return (focal_w * ce_loss).mean()


# Highly imbalanced dataset: class 0=90%, classes 1-4=10% total
N_IMBAL = 1000
X_imb   = torch.randn(N_IMBAL, 16, device=device)
y_imb   = torch.zeros(N_IMBAL, dtype=torch.long, device=device)
minority_idx = torch.randperm(N_IMBAL)[:100]
y_imb[minority_idx] = torch.randint(1, 5, (100,), device=device)
print(f'Class distribution: {torch.bincount(y_imb).tolist()}')

imb_ds  = TensorDataset(X_imb[:800], y_imb[:800])
imb_val = TensorDataset(X_imb[800:], y_imb[800:])
imb_trl = DataLoader(imb_ds,  batch_size=32, shuffle=True)
imb_vll = DataLoader(imb_val, batch_size=64)


def make_classifier(n_out=5):
    return nn.Sequential(
        nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, n_out)
    ).to(device)


# Class weights inversely proportional to frequency
counts = torch.bincount(y_imb).float()
alpha  = (counts.sum() / (len(counts) * counts)).to(device)

for loss_name, loss_obj in [
    ('CE (no weighting)', nn.CrossEntropyLoss()),
    ('CE + class weights', nn.CrossEntropyLoss(weight=alpha)),
    ('Focal (gamma=2)',    FocalLoss(gamma=2.0, alpha=alpha)),
]:
    m   = make_classifier()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(30):
        m.train()
        for xb, yb in imb_trl:
            opt.zero_grad(); loss_obj(m(xb), yb).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        preds = torch.cat([m(xb).argmax(1) for xb, _ in imb_vll])
        trues = torch.cat([yb for _, yb in imb_vll])
    minority_mask = (trues > 0)
    if minority_mask.sum() > 0:
        minority_acc = (preds[minority_mask] == trues[minority_mask]).float().mean().item()
    else:
        minority_acc = 0.0
    overall_acc = (preds == trues).float().mean().item()
    print(f'{loss_name:<25}  overall={overall_acc:.3f}  minority={minority_acc:.3f}')


## Real-World Example 2: Contrastive Loss for Metric Learning

In [ ]:
# Contrastive loss trains embeddings to pull similar pairs together
# and push dissimilar pairs apart (Chopra et al. 2005).
# L = (1-y) * 0.5 * d^2 + y * 0.5 * max(0, margin - d)^2
# y=0: same class; y=1: different class
# Used in: face verification, signature matching, duplicate detection.

torch.manual_seed(5)


class ContrastiveLoss(nn.Module):
    '''Contrastive loss for metric learning.
    Args:
        margin: minimum distance required between dissimilar pairs
    '''
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        '''Args:
           emb1, emb2: embeddings of shape (B, D)
           labels: 0=same class, 1=different class  shape (B,)
        '''
        dist = F.pairwise_distance(emb1, emb2, p=2)
        same_loss  = (1 - labels) * 0.5 * dist.pow(2)
        diff_loss  = labels * 0.5 * F.relu(self.margin - dist).pow(2)
        return (same_loss + diff_loss).mean()


class SiameseEncoder(nn.Module):
    '''Shared encoder for Siamese network.'''
    def __init__(self, in_dim=32, emb_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, emb_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=-1)  # L2-normalise


# Generate pair dataset: (x1, x2, label=0 if same class, 1 if different)
N_CLASSES_C = 4
N_PAIRS     = 800
X_enc = torch.randn(200, 32, device=device)
y_enc = torch.randint(0, N_CLASSES_C, (200,), device=device)

pairs_i, pairs_j, pair_labels = [], [], []
for _ in range(N_PAIRS):
    i, j = torch.randint(0, 200, (2,)).tolist()
    pairs_i.append(i); pairs_j.append(j)
    pair_labels.append(0 if y_enc[i] == y_enc[j] else 1)

i_t = torch.tensor(pairs_i)
j_t = torch.tensor(pairs_j)
l_t = torch.tensor(pair_labels, dtype=torch.float, device=device)
pair_ds  = TensorDataset(i_t[:640], j_t[:640], l_t[:640])
pair_val = TensorDataset(i_t[640:], j_t[640:], l_t[640:])
pair_trl = DataLoader(pair_ds,  batch_size=32, shuffle=True)
pair_vll = DataLoader(pair_val, batch_size=64)

encoder    = SiameseEncoder().to(device)
opt_enc    = torch.optim.Adam(encoder.parameters(), lr=1e-3)
cont_loss  = ContrastiveLoss(margin=1.0)

for _ in range(30):
    encoder.train()
    for bi, bj, bl in pair_trl:
        opt_enc.zero_grad()
        e1 = encoder(X_enc[bi])
        e2 = encoder(X_enc[bj])
        cont_loss(e1, e2, bl).backward()
        opt_enc.step()

encoder.eval()
with torch.no_grad():
    dists = []
    lbls  = []
    for bi, bj, bl in pair_vll:
        d = F.pairwise_distance(encoder(X_enc[bi]), encoder(X_enc[bj]))
        dists.extend(d.tolist()); lbls.extend(bl.tolist())
dists = np.array(dists); lbls = np.array(lbls)
same_dists = dists[lbls == 0]
diff_dists = dists[lbls == 1]
print(f'Same-class pairs:  mean dist = {same_dists.mean():.4f} ± {same_dists.std():.4f}')
print(f'Diff-class pairs:  mean dist = {diff_dists.mean():.4f} ± {diff_dists.std():.4f}')
print('Good embeddings: small same-class distance, large diff-class distance.')


## Real-World Example 3: Compound Loss with Learnable alpha/beta/gamma Weights

In [ ]:
# Multi-task and multi-objective training often combines several losses.
# Naive approach: manually tune alpha, beta, gamma weights (expensive).
# Better: learn the loss weights as parameters (Kendall et al. 2018
# 'Multi-Task Learning Using Uncertainty to Weigh Losses').
# L = sum_i  (1 / (2 * sigma_i^2)) * L_i + log(sigma_i)

torch.manual_seed(6)


class MultiTaskNet(nn.Module):
    '''Shared trunk with two heads: regression + classification.'''
    def __init__(self):
        super().__init__()
        self.trunk    = nn.Sequential(
            nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU()
        )
        self.reg_head = nn.Linear(32, 1)    # regression output
        self.cls_head = nn.Linear(32, 4)    # classification output (4 classes)

    def forward(self, x):
        feat = self.trunk(x)
        return self.reg_head(feat), self.cls_head(feat)


class LearnedLossWeights(nn.Module):
    '''Uncertainty-weighted compound loss (Kendall et al.).
    log_sigma2 are learnable; act as automatic loss balancers.
    '''
    def __init__(self, n_tasks=3):
        super().__init__()
        # log(sigma^2) initialised to 0 (sigma=1, equal weighting)
        self.log_sigma2 = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses):
        '''losses: list of scalar task losses'''
        total = 0.0
        for i, l in enumerate(losses):
            # weight = 1/(2*sigma^2); penalty = log(sigma)
            weight  = torch.exp(-self.log_sigma2[i])
            total  += weight * l + 0.5 * self.log_sigma2[i]
        return total


# Multi-task dataset
N_MT  = 800
X_mt  = torch.randn(N_MT, 16, device=device)
y_reg_mt = (X_mt[:, 0] + X_mt[:, 1] + 0.1 * torch.randn(N_MT, device=device)).unsqueeze(1)
y_cls_mt = torch.randint(0, 4, (N_MT,), device=device)
mt_ds  = TensorDataset(X_mt[:640], y_reg_mt[:640], y_cls_mt[:640])
mt_val = TensorDataset(X_mt[640:], y_reg_mt[640:], y_cls_mt[640:])
mt_trl = DataLoader(mt_ds,  batch_size=32, shuffle=True)
mt_vll = DataLoader(mt_val, batch_size=64)

mt_net     = MultiTaskNet().to(device)
loss_weighter = LearnedLossWeights(n_tasks=2).to(device)
all_params = list(mt_net.parameters()) + list(loss_weighter.parameters())
opt_mt     = torch.optim.Adam(all_params, lr=1e-3)
mse_fn     = nn.MSELoss()
ce_fn      = nn.CrossEntropyLoss()

for _ in range(40):
    mt_net.train(); loss_weighter.train()
    for xb, yrb, ycb in mt_trl:
        opt_mt.zero_grad()
        r_out, c_out = mt_net(xb)
        l_reg = mse_fn(r_out, yrb)
        l_cls = ce_fn(c_out, ycb)
        total = loss_weighter([l_reg, l_cls])
        total.backward(); opt_mt.step()

mt_net.eval(); loss_weighter.eval()
sigma2_vals = torch.exp(loss_weighter.log_sigma2).detach().cpu().numpy()
print(f'Learned sigma^2 for regression task: {sigma2_vals[0]:.4f}')
print(f'Learned sigma^2 for classification task: {sigma2_vals[1]:.4f}')
print('Larger sigma^2 = less weight on that task (network found it harder/noisier).')

with torch.no_grad():
    val_reg_mse = sum(
        mse_fn(mt_net(xb)[0], yrb).item() * len(xb)
        for xb, yrb, _ in mt_vll
    ) / len(mt_val)
    val_cls_correct = sum(
        (mt_net(xb)[1].argmax(1) == ycb).sum().item()
        for xb, _, ycb in mt_vll
    )
print(f'Val regression MSE:        {val_reg_mse:.4f}')
print(f'Val classification accuracy: {val_cls_correct/len(mt_val):.4f}')
print('Learned weighting automatically balances regression and classification objectives.')
